# 1. The Anveshar pipeline harness

A reproducible, provenance-tracked pipeline that turns a rare cancer into a cited, confidence-scored cross-condition translation dossier. Five auditable stages: resolve, databases, target validation, translate, assemble. Offline is deterministic; `live=True` queries Open Targets, Ensembl, ChEMBL, PubMed, and DepMap.

In [ ]:
import os, sys
def _find_repo():
    p = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(p, "anveshar", "__init__.py")):
            return p
        p = os.path.dirname(p)
    return None
REPO = _find_repo()
if REPO and REPO not in sys.path:
    sys.path.insert(0, REPO)
import anveshar
print("anveshar loaded from", os.path.dirname(anveshar.__file__))

## Run the harness offline (deterministic)

In [ ]:
from anveshar.pipeline import run
d = run("renal medullary carcinoma", live=False)
print(d["condition"], "| drivers:", d["drivers"], "| mode:", d["mode"])
s = d["summary"]; bc = s["best_confidence"]
print("actionable:", s["n_actionable"], "| tissue-agnostic:", s["n_tissue_agnostic"],
      "| cross-condition:", s["n_cross_condition"])
print("best confidence:", bc)

## Provenance: every stage is auditable

In [ ]:
import pandas as pd
pd.DataFrame(d["provenance"])

## Borrowable approved therapies (cited, confidence-scored)

In [ ]:
pd.DataFrame([{"gene": m["gene"], "drug": m["drug"], "approved_in": m["approved_in"],
               "tier": m["tier"], "citation": m["citation"].get("url", "")}
              for m in d["translation"]["actionable"]])

## The bespoke, driver-class-tailored comp-bio workflow

In [ ]:
w = d["workflow"]
print("driver class:", w["driver_class"], "\n")
for i, st in enumerate(w["steps"], 1):
    print(str(i) + ". " + st["title"])
    print("   tools:", st["tools"])
    print("   checkpoint:", st["outputs"])
    print()

## Induced dependency (synthetic lethality) for loss drivers

In [ ]:
d.get("induced_dependencies", {})

## Optional: run live (Open Targets + DepMap functional genomics)
Requires internet; wrapped so a rate limit does not break the notebook.

In [ ]:
try:
    dl = run("renal medullary carcinoma", live=True)
    tv = dl.get("target_validation", {})
    display(pd.DataFrame([{"gene": g, "score": v.get("score"), "verdict": v.get("verdict"),
                           "tractability_SM": v.get("sm_tractability"), "selective": v.get("selective_dependency")}
                          for g, v in tv.items() if v.get("available")]))
except Exception as e:
    print("live stage skipped:", type(e).__name__, e)

---
*Disclaimer: this notebook produces research and educational analysis of public data, not medical advice. Confidence and validation scores summarize evidence strength, not the probability of benefit for any individual. Every clinical decision must be made by a qualified health care provider, ideally within a clinical trial.*

*Developed by Dig Vijay Kumar Yarlagadda, [digvijayky.com](https://digvijayky.com).*